# Issue #44: Leakage-Safe Model Inputs

This notebook validates and records the approved preprocessing boundary for Ridge and histogram-gradient boosting. It does not train, compare, or select candidate models.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

# Local use: leave as None. Colab use: set this to your own Drive checkout.
REPO_ROOT_OVERRIDE = None  # Example: Path('/content/drive/MyDrive/<your-folder>')

if REPO_ROOT_OVERRIDE is not None:
    repo_root = Path(REPO_ROOT_OVERRIDE).expanduser().resolve()
else:
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    repo_root = next(path for path in candidates if (path / 'data' / 'allstate_claims_data.csv').exists())

issue_dir = repo_root / 'notebooks' / 'ragib-nehal' / 'oct-notebooks' / 'leakage-safe-model-inputs'
sys.path.insert(0, str(issue_dir))
from preprocessing_pipeline import generate_preprocessing_artifacts
print('Repository root located; frozen inputs are available.')

Repository root located; frozen inputs are available.


## Regenerate compact evidence

The reusable module verifies the frozen hashes, removes protected test rows before modeling objects exist, fits both preprocessors on training only, and writes compact review artifacts.

In [2]:
receipt = generate_preprocessing_artifacts(repo_root, issue_dir)
print(f"Generated run: {receipt['run_id']}")
print(f"Fit scope: {receipt['fitting_scope']}")

Generated run: OCT-PREP-20261027T072126Z-63abee69
Fit scope: train only; validation transform only


## Contract checks

In [3]:
checks = pd.read_csv(issue_dir / 'preprocessing_checks.csv')
display(checks[['check_id', 'description', 'status']])

,check_id,description,status
0,source_hash,Source hash matches frozen OCT-MD-001,pass
1,manifest_hash,Manifest hash matches frozen OCT-SPLIT-001,pass
2,partition_counts,Train and validation counts match the frozen s...,pass
3,test_partition_protected,Returned modeling object exposes no test rows,pass
4,feature_contract,Inputs are exactly 116 categorical plus 14 con...,pass
5,train_only_fit,All fitted preprocessing state used training r...,pass
6,category_unseen_counts,Validation-only categories are counted without...,pass
7,category_rare_or_unseen_cells,Validation rare-or-unseen cell count matches a...,pass
8,ridge_matrix,Ridge matrices align and contain only finite v...,pass
9,tree_matrix,"Tree matrices preserve mask, cardinality, and ...",pass


## Category routing evidence

These counts use features only; validation loss is not used.

In [4]:
category_summary = pd.read_csv(issue_dir / 'category_handling_summary.csv')
display(category_summary.loc[category_summary['validation_unseen_cells'] > 0])
print('validation_unseen_cells:', int(category_summary['validation_unseen_cells'].sum()))
print('validation_rare_or_unseen_cells:', int(category_summary['validation_rare_or_unseen_cells'].sum()))

,feature,train_levels,frequent_train_levels,rare_train_levels,rare_train_cells,validation_unseen_cells,validation_rare_or_unseen_cells,has_rare_bucket,tree_cardinality_after_pooling
104,cat105,18,9,9,347,1,65,True,10
109,cat110,128,39,89,2048,1,419,True,40
112,cat113,60,41,19,276,1,42,True,42
115,cat116,306,104,202,4591,6,972,True,105


validation_unseen_cells: 9
validation_rare_or_unseen_cells: 2560


## Both approved model families

In [5]:
schema = json.loads((issue_dir / 'feature_schema.json').read_text())
family_summary = pd.DataFrame([
    {'model_family': name, **details}
    for name, details in schema['model_families'].items()
])
display(family_summary)

,model_family,categorical,continuous,sparse_output,categorical_mask
0,ridge,pooled one-hot,train-median imputation then StandardScaler,True,NaN
1,hist_gradient_boosting,pooled ordinal codes,train-median imputation without scaling,NaN,"[True, True, True, True, True, True, True, Tru..."


## Pipeline-readiness baseline reproduction

These are the already accepted constant baselines, recomputed only as a readiness check. They are not feature-model results.

In [6]:
baseline_summary = pd.DataFrame(receipt['baseline_reproduction']).T
display(baseline_summary)
print('Issue #45 owns candidate model scoring and comparison.')

,fitted_constant,validation_mae
training_mean_constant,3038.840255,1962.918654
training_median_constant,2115.390000,1805.086353


Issue #45 owns candidate model scoring and comparison.
